In [1]:
!nvidia-smi

Mon Apr 20 17:27:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

Torch version: 2.10.0+cu128
CUDA available: True
GPU name: NVIDIA A100-SXM4-40GB


In [ ]:
import os, getpass
token = getpass.getpass("GitHub token: ")
!git clone https://{token}@github.com/jasmineztruong8/efficient-codegen.git
%cd efficient-codegen

GitHub token: ··········
Cloning into 'efficient-codegen'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 157 (delta 4), reused 11 (delta 2), pack-reused 140 (from 1)
Receiving objects: 100% (157/157), 15.67 MiB | 21.98 MiB/s, done.
Resolving deltas: 100% (62/62), done.
/content/efficient-codegen


In [ ]:
!git fetch origin
!git checkout model-skeleton
!git pull origin model-skeleton
!git log --oneline -n 10

Already on 'model-skeleton'
Your branch is up to date with 'origin/model-skeleton'.
From https://github.com/jasmineztruong8/efficient-codegen
 * branch            model-skeleton -> FETCH_HEAD
Already up to date.
879d3b9 (HEAD -> model-skeleton, origin/model-skeleton) Merge pull request #6 from jasmineztruong8/profiling-pipeline
49be067 Merge pull request #4 from jasmineztruong8/jg/training-data-construction
cd3be46 now all generated candidates are being evaluated at Pass@1 and Pass@5
a691585 update evaluated candidates full dataset
b852573 fixing bug in evaluate_candidates.py
965fa6e (origin/jg/training-data-construction) Smoke test updates
44f4bcd add evaluated candidates full dataset
9c80655 (jg/training-data-construction) Generate runtime aware and control datasets from benchmarked candidates
1c5237e Merge pull request #3 from jasmineztruong8/profiling-pipeline
924867a Add generated candidates full dataset and benchmarked full candidates dataset


In [ ]:
!ls training/data
!wc -l training/data/control.jsonl training/data/runtime_aware.jsonl

control.jsonl  runtime_aware.jsonl
   2110 training/data/control.jsonl
   2110 training/data/runtime_aware.jsonl
   4220 total


In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes wandb trl torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 56.7 MB/s eta 0:00:00


In [ ]:
%cd /content/efficient-codegen
!ls training/data

/content/efficient-codegen
control.jsonl  runtime_aware.jsonl


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!mkdir -p /content/drive/MyDrive/efficient-codegen/checkpoints

In [ ]:
!python training/train.py \
  --mode control \
  --data_path training/data/control.jsonl \
  --output_dir /content/drive/MyDrive/efficient-codegen/checkpoints/control_smoke \
  --limit 100 \
  --num_train_epochs 1 \
  --per_device_train_batch_size 2 \
  --gradient_accumulation_steps 2

Loading weights: 100% 338/338 [00:01<00:00, 274.43it/s, Materializing param=model.norm.weight]
Training on 100 examples  (mode=control)
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Adding EOS to train dataset: 100% 100/100 [00:00<00:00, 19353.56 examples/s]
Tokenizing train dataset: 100% 100/100 [00:00<00:00, 1153.98 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
{'loss': '1.103', 'grad_norm': '0.7617', 'learning_rate': '0.0001577', 'entropy': '0.9053', 'num_tokens': '9054', 'mean_token_accuracy': '0.7723', 'epoch': '0.4'}
{'loss': '0.5991', 'grad_norm': '0.8828', 'learning_rate': '3.174e-05', 'entropy': '0.6101', 'num_tokens': '1.791e+04', 'mean_token_accuracy': '0.8542', 'epoch': '0.8'}
{'train_runtime': '29.64', '

In [ ]:
!python training/train.py \
  --mode runtime_aware \
  --data_path training/data/runtime_aware.jsonl \
  --output_dir /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_smoke \
  --limit 100 \
  --num_train_epochs 1 \
  --per_device_train_batch_size 2 \
  --gradient_accumulation_steps 2

Loading weights: 100% 338/338 [00:01<00:00, 293.36it/s, Materializing param=model.norm.weight]
Training on 100 examples  (mode=runtime_aware)
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Adding EOS to train dataset: 100% 100/100 [00:00<00:00, 18479.55 examples/s]
Tokenizing train dataset: 100% 100/100 [00:00<00:00, 1168.34 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
{'loss': '1.121', 'grad_norm': '0.957', 'learning_rate': '0.0001577', 'entropy': '0.9205', 'num_tokens': '8829', 'mean_token_accuracy': '0.7703', 'epoch': '0.4'}
{'loss': '0.6087', 'grad_norm': '0.8594', 'learning_rate': '3.174e-05', 'entropy': '0.6222', 'num_tokens': '1.749e+04', 'mean_token_accuracy': '0.8507', 'epoch': '0.8'}
{'train_runtime': '29.8

In [ ]:
!ls /content/drive/MyDrive/efficient-codegen/checkpoints/control_smoke
!ls /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_smoke

adapter_config.json	   checkpoint-25	  tokenizer.json
adapter_model.safetensors  README.md		  training_args.bin
chat_template.jinja	   tokenizer_config.json
adapter_config.json	   checkpoint-25	  tokenizer.json
adapter_model.safetensors  README.md		  training_args.bin
chat_template.jinja	   tokenizer_config.json


In [ ]:
!mkdir -p /content/drive/MyDrive/efficient-codegen/logs

In [ ]:
import wandb
wandb.login()

True

In [ ]:
!python training/train.py \
  --mode control \
  --data_path training/data/control.jsonl \
  --output_dir /content/drive/MyDrive/efficient-codegen/checkpoints/control_full \
  --use_wandb \
  --wandb_project hpml-efficient-codegen

Loading weights: 100% 338/338 [00:01<00:00, 284.44it/s, Materializing param=model.norm.weight]
Training on 2110 examples  (mode=control)
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Adding EOS to train dataset: 100% 2110/2110 [00:00<00:00, 34265.73 examples/s]
Tokenizing train dataset: 100% 2110/2110 [00:01<00:00, 1732.04 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: jg4553 (jg4553-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.25.

In [ ]:
!ls /content/drive/MyDrive/efficient-codegen/checkpoints/control_full

adapter_config.json	   checkpoint-264	  tokenizer.json
adapter_model.safetensors  checkpoint-396	  training_args.bin
chat_template.jinja	   README.md
checkpoint-132		   tokenizer_config.json


In [ ]:
!python training/train.py \
  --mode runtime_aware \
  --data_path training/data/runtime_aware.jsonl \
  --output_dir /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_full \
  --use_wandb \
  --wandb_project hpml-efficient-codegen

Loading weights: 100% 338/338 [00:01<00:00, 292.91it/s, Materializing param=model.norm.weight]
Training on 2110 examples  (mode=runtime_aware)
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Adding EOS to train dataset: 100% 2110/2110 [00:00<00:00, 34051.88 examples/s]
Tokenizing train dataset: 100% 2110/2110 [00:01<00:00, 1715.27 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: jg4553 (jg4553-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: ⢿ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved loca

In [ ]:
!python training/evaluate_model.py \
  --model_path Qwen/Qwen2.5-Coder-1.5B-Instruct \
  --run_name base_slm \
  --data_path data/curated/test/dataset_clean.json \
  --use_wandb

Loading dataset from data/curated/scale1k/dataset_clean.json
Evaluating on 1000 problems
Loading model: Qwen/Qwen2.5-Coder-1.5B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:00<00:00, 427.63it/s, Materializing param=model.norm.weight]
Generating: 100% 125/125 [14:49<00:00,  7.12s/it]
Evaluating correctness and benchmarking...
Evaluating: 100% 1000/1000 [00:50<00:00, 19.90it/s]

Run: base_slm
Problems evaluated:           1000
Problems with ≥1 passing:     468 / 1000
Pass@1 (unbiased):            0.3690
Median execution time (s):    0.000089
Avg generation latency (s):   0.89
Peak CUDA memory (MB):        6691.5
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: jg4553 (jg4553-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: Tracking run wit

In [ ]:
!python3 training/evaluate_model.py \
  --model_path /content/drive/MyDrive/efficient-codegen/checkpoints/control_full \
  --base_model_name Qwen/Qwen2.5-Coder-1.5B-Instruct \
  --run_name control_sft \
  --data_path data/curated/test/dataset_clean.json \
  --use_wandb

Loading dataset from data/curated/scale1k/dataset_clean.json
Evaluating on 1000 problems
Detected PEFT checkpoint at /content/drive/MyDrive/efficient-codegen/checkpoints/control_full
Loading base model: Qwen/Qwen2.5-Coder-1.5B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:00<00:00, 429.51it/s, Materializing param=model.norm.weight]
Generating: 100% 125/125 [12:32<00:00,  6.02s/it]
Evaluating correctness and benchmarking...
Evaluating: 100% 1000/1000 [01:55<00:00,  8.67it/s]

Run: control_sft
Problems evaluated:           1000
Problems with ≥1 passing:     751 / 1000
Pass@1 (unbiased):            0.7090
Median execution time (s):    0.000091
Avg generation latency (s):   0.75
Peak CUDA memory (MB):        6691.5
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: jg4553 (jg4553-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]

In [ ]:
!python3 training/evaluate_model.py \
  --model_path /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_full \
  --base_model_name Qwen/Qwen2.5-Coder-1.5B-Instruct \
  --run_name runtime_aware_sft \
  --data_path data/curated/test/dataset_clean.json \
  --use_wandb

Loading dataset from data/curated/scale1k/dataset_clean.json
Evaluating on 1000 problems
Detected PEFT checkpoint at /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_full
Loading base model: Qwen/Qwen2.5-Coder-1.5B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:00<00:00, 420.83it/s, Materializing param=model.norm.weight]
Generating: 100% 125/125 [12:22<00:00,  5.94s/it]
Evaluating correctness and benchmarking...
Evaluating: 100% 1000/1000 [01:50<00:00,  9.05it/s]

Run: runtime_aware_sft
Problems evaluated:           1000
Problems with ≥1 passing:     745 / 1000
Pass@1 (unbiased):            0.7016
Median execution time (s):    0.000090
Avg generation latency (s):   0.74
Peak CUDA memory (MB):        6691.5
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: jg4553 (jg4553-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relog

# Ablation and Profiling Runs

---



In [1]:
!nvidia-smi
import os
print(os.getcwd())
!ls

Mon Apr 20 18:09:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import wandb
wandb.login()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: jg4553 (jg4553-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import os, getpass
token = getpass.getpass("GitHub token: ")
!git clone https://{token}@github.com/jasmineztruong8/efficient-codegen.git
%cd /content/efficient-codegen
!git fetch origin
!git checkout main
!git pull origin main
!git log --oneline -n 10

GitHub token: ··········
fatal: destination path 'efficient-codegen' already exists and is not an empty directory.
/content/efficient-codegen
Already on 'main'
Your branch is up to date with 'origin/main'.
From https://github.com/jasmineztruong8/efficient-codegen
 * branch            main       -> FETCH_HEAD
Already up to date.
1ba36ea (HEAD -> main, origin/main, origin/HEAD) Merge pull request #5 from jasmineztruong8/amahajan-train-run
b4ec13a (origin/amahajan-train-run) add requirements
2d20775 Merge pull request #9 from jasmineztruong8/model-skeleton
67ef2af (origin/model-skeleton) Merge pull request #8 from jasmineztruong8/profiling-pipeline
ae9ce66 (origin/profiling-pipeline) update README with new profiling results: 320-sample batch=64 run, add CPU-GPU sync bottleneck
d9b6dbb Adding wandb team entity flag
22c637e Merge pull request #7 from jasmineztruong8/profiling-pipeline
3f4cccc update directory structure
fb95378 Remove unused extract_profiling_20.py (legacy script)
8f532ee up

In [4]:
!pip install -q transformers datasets accelerate peft bitsandbytes wandb trl torch

In [5]:
!ls
!ls training
!ls training/data

data	   generation  outputs	  README.md	    scripts   wandb
execution  notebooks   profiling  requirements.txt  training
data  evaluate_model.py  select_training_data.py  train.py
control.jsonl  runtime_aware.jsonl


In [9]:
!python3 training/evaluate_model.py \
  --model_path /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_full \
  --base_model_name Qwen/Qwen2.5-Coder-1.5B-Instruct \
  --run_name runtime_aware_sft_profiled_smoke \
  --data_path data/curated/validation/dataset_clean.json \
  --limit 100 \
  --output_path outputs/runtime_aware_sft_profiled_smoke.jsonl \
  --profile \
  --trace_dir outputs/tb_profiler/runtime_aware_smoke \
  --use_wandb

Loading dataset from data/curated/scale1k/dataset_clean.json
Evaluating on 100 problems
Detected PEFT checkpoint at /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_full
Loading base model: Qwen/Qwen2.5-Coder-1.5B-Instruct
config.json: 100% 660/660 [00:00<00:00, 4.10MB/s]
tokenizer_config.json: 7.30kB [00:00, 17.4MB/s]
vocab.json: 2.78MB [00:00, 94.4MB/s]
merges.txt: 1.67MB [00:00, 102MB/s]
tokenizer.json: 7.03MB [00:00, 145MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 3.09G/3.09G [00:09<00:00, 335MB/s]
Loading weights: 100% 338/338 [00:00<00:00, 408.00it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 242/242 [00:00<00:00, 1.44MB/s]
Profiling enabled — traces will be saved to outputs/tb_profiler/runtime_aware_smoke
Generating:   0% 0/13 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from th

In [10]:
!python3 training/evaluate_model.py \
  --model_path /content/drive/MyDrive/efficient-codegen/checkpoints/control_full \
  --base_model_name Qwen/Qwen2.5-Coder-1.5B-Instruct \
  --run_name control_sft_profiled \
  --data_path data/curated/validation/dataset_clean.json \
  --output_path outputs/control_sft_profiled.jsonl \
  --profile \
  --trace_dir outputs/tb_profiler/control \
  --use_wandb

Loading dataset from data/curated/scale1k/dataset_clean.json
Evaluating on 1000 problems
Detected PEFT checkpoint at /content/drive/MyDrive/efficient-codegen/checkpoints/control_full
Loading base model: Qwen/Qwen2.5-Coder-1.5B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:00<00:00, 426.91it/s, Materializing param=model.norm.weight]
Profiling enabled — traces will be saved to outputs/tb_profiler/control
Generating:   0% 0/125 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
Generating: 100% 125/125 [18:47<00:00,  9.02s/it]
Profiler traces saved to outputs/tb_profiler/control
Evaluating correctness and benchmarking...
Evaluating: 100% 1000/1000 [01:51<00:00,  8.93it/s]

Run: control_sft_profiled
Problems evaluated:     

In [6]:
!python3 training/evaluate_model.py \
  --model_path /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_full \
  --base_model_name Qwen/Qwen2.5-Coder-1.5B-Instruct \
  --run_name runtime_aware_sft_profiled \
  --data_path data/curated/validation/dataset_clean.json \
  --output_path outputs/runtime_aware_sft_profiled.jsonl \
  --profile \
  --trace_dir outputs/tb_profiler/runtime_aware \
  --use_wandb


Loading dataset from data/curated/scale1k/dataset_clean.json
Evaluating on 1000 problems
Detected PEFT checkpoint at /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_full
Loading base model: Qwen/Qwen2.5-Coder-1.5B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:00<00:00, 424.20it/s, Materializing param=model.norm.weight]
Profiling enabled — traces will be saved to outputs/tb_profiler/runtime_aware
Generating:   0% 0/125 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
Generating: 100% 125/125 [19:33<00:00,  9.38s/it]
Profiler traces saved to outputs/tb_profiler/runtime_aware
Evaluating correctness and benchmarking...
Evaluating: 100% 1000/1000 [01:52<00:00,  8.89it/s]

Run: runtime_aware_sft_profiled


In [7]:
!python3 training/evaluate_model.py \
  --model_path Qwen/Qwen2.5-Coder-1.5B-Instruct \
  --run_name base_slm_profiled \
  --data_path data/curated/validation/dataset_clean.json \
  --output_path outputs/base_slm_profiled.jsonl \
  --profile \
  --trace_dir outputs/tb_profiler/base \
  --use_wandb

Loading dataset from data/curated/scale1k/dataset_clean.json
Evaluating on 1000 problems
Loading model: Qwen/Qwen2.5-Coder-1.5B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:00<00:00, 425.48it/s, Materializing param=model.norm.weight]
Profiling enabled — traces will be saved to outputs/tb_profiler/base
Generating:   0% 0/125 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
Generating: 100% 125/125 [23:59<00:00, 11.51s/it]
Profiler traces saved to outputs/tb_profiler/base
Evaluating correctness and benchmarking...
Evaluating: 100% 1000/1000 [00:25<00:00, 39.76it/s]

Run: base_slm_profiled
Problems evaluated:           1000
Problems with ≥1 passing:     472 / 1000
Pass@1 (unbiased):            0.3656
Median execution ti